# $(i, i+2)$ hopping interaction

We need to implement the next to nearest neighbours interaction for the hopping term.
Let us try three strategies:
1. do a simple $(i,i+1)$ interaction with the correct parameters and insert _a posteriori_ an identity MPO
2. make two sequential svds to obtain a three-site mpo (careful with the canonical form).
3. use the swap gate technique implemented by E. M. Stoudenmire and Steven R. White

In [7]:
%load_ext autoreload
%autoreload 2

import numpy as np

from ncon import ncon

from scipy.sparse import csc_array, kron
import scipy.sparse as sp
from scipy.linalg import svd, norm

from qs_mps.sparse_hamiltonians_and_operators import diagonalization, U_evolution_sparse
from qs_mps.mps_class import MPS
from qs_mps.utils import tensor_shapes, mps_to_vector, logarithm_base_d, mpo_to_matrix

import matplotlib.pyplot as plt

# default parameters of the plot layout
plt.rcParams["text.usetex"] = True  # use latex
plt.rcParams["font.size"] = 13
plt.rcParams["figure.dpi"] = 300
plt.rcParams["figure.constrained_layout.use"] = True

In [4]:
def swap_gate(d):
    W = np.zeros((d,d,d,d), dtype=int)
    for i in range(d):
        for j in range(d):
            W[j,i,i,j] = 1
    return W

swap_mpo = swap_gate(3)

In [6]:
def id_gate(d,chi):
    I = np.zeros((chi, chi, d, d))

    for a in range(chi):
        for s in range(d):
            I[a,a,s,s] = 1.0
    return I

I_mpo = id_gate(3,9)

In [8]:
# Single-site identity
Id = sp.identity(3, format="csr").toarray()
O = csc_array((3, 3), dtype=complex).toarray()

# Spin operators
Sz = (1/2) * sp.diags([1, 0, -1], 0, format="csr")

S_plus  = sp.csr_matrix([[0, 0, 1],
                         [0, 0, 0],
                         [0, 0, 0]])

S_minus = sp.csr_matrix([[0, 0, 0],
                         [0, 0, 0],
                         [1, 0, 0]])

# Hole hopping operators

# hole goes into a spin up state
T_up_h   = sp.csr_matrix([[0, 1, 0],
                          [0, 0, 0],
                          [0, 0, 0]])

# hole goes into a spin down state
T_down_h = sp.csr_matrix([[0, 0, 0],
                          [0, 0, 0],
                          [0, 1, 0]])

# spin up goes into a hole state
T_h_up   = sp.csr_matrix([[0, 0, 0],
                          [1, 0, 0],
                          [0, 0, 0]])

# spin down goes into a hole state
T_h_down = sp.csr_matrix([[0, 0, 0],
                          [0, 0, 1],
                          [0, 0, 0]])

# Hole number operator
n_h = sp.csr_matrix([[0, 0, 0],
                     [0, 1, 0],
                     [0, 0, 0]])

In [10]:
def U_i_ip2_tJ(tp_up, tp_down, delta):
    """
    U_i_ip2

    This function computes the exponential of the 2-site hamiltonian for the t-J model.
    It returns to versions: 
    1. one with a time step delta/2 to use at the initial and final step
    of the trotterization
    2. one with a time step delta to use in the bulk steps of the trotterization

    """

    # choose the hamiltonian parameters
    H_i_ip2 = (
    - (tp_up/8) * kron(T_up_h, T_h_up) 
    - (tp_up/8) * kron(T_h_up, T_up_h)
    - (tp_down/8) * kron(T_down_h, T_h_down) 
    - (tp_down/8) * kron(T_h_down, T_down_h)).toarray()

    # initial and final 2-site evolution operator
    op_ev_if = sp.linalg.expm(-1j * delta/2 * H_i_ip2)

    # bulk 2-site evolution operator
    op_ev_bulk = sp.linalg.expm(-1j * delta * H_i_ip2)
    
    return op_ev_if, op_ev_bulk

def evolution_mpo_svd_1_tJ(op_ev: np.ndarray, d: int=3, schmidt_tol: float=1e-15, trunc: bool=False):
    """
    evolution_mpo_svd

    This function takes the edges, and bulk 2-site evolution operators (of the t-J model) and performs an svd
    to separate the matrix into site i and site i+1. Reshaping the results of the svd
    we can obtain the mpo for those evolution operators (with bounded bond dimension D<=d^2)

    """
    op_ev = op_ev.reshape(d,d,d,d)
    op_ev = op_ev.transpose(0,2,1,3)
    op_ev = op_ev.reshape(d*d,d*d)

    u, s, v = svd(op_ev, full_matrices=False)

    if trunc:
        condition = s >= schmidt_tol
        s_trunc = np.extract(condition, s)
        s = s_trunc
        v = v[:len(s),:]

    site_i = u.reshape(d,d,u.shape[1])
    site_i = site_i[:, :, :len(s)]
    site_i = site_i.transpose(2,0,1)
    site_i = site_i.reshape(1,len(s),d,d)

    site_ip1 = ncon([np.diag(s), v],[[-1, 1],[1, -2]]).reshape(v.shape[0],d,d)
    site_ip1 = site_ip1.reshape(1,v.shape[0],d,d)
    site_ip1 = site_ip1.transpose(1,0,2,3)


    tol = 1e-15 * np.max(np.abs(site_i))
    site_i.real[np.abs(site_i.real) < tol] = 0
    site_i.imag[np.abs(site_i.imag) < tol] = 0
    
    tol = 1e-15 * np.max(np.abs(site_ip1))
    site_ip1.real[np.abs(site_ip1.real) < tol] = 0
    site_ip1.imag[np.abs(site_ip1.imag) < tol] = 0
    
    return site_i, site_ip1

In [12]:
half_chain_length = 3
tp_up, tp_down = 1, 1
trotter_steps = 100
final_time = 2
delta = final_time/trotter_steps
n = 2*half_chain_length + 1

In [ ]:
op_ev_half, op_ev_delta = U_i_ip2_tJ(tp_up, tp_down, delta)
site_i_half, site_ip2_half = evolution_mpo_svd_1_tJ(op_ev_half)
site_i_delta, site_ip2_delta = evolution_mpo_svd_1_tJ(op_ev_delta)
tensor_shapes([site_i_half, site_ip2_half])
id_enlarged_mpo = id_gate(3,site_i_half.shape[1])

(1, 9, 3, 3)
(9, 1, 3, 3)


In [ ]:
def parity_groups(n):
    even_A = [[i for i in range(n) if (i%4 == 0)]]
    even_C = [[i for i in range(n) if (i%4 == 1)]]
    odd_B = [[i for i in range(n) if (i%4 == 3)]]
    odd_D = [[i for i in range(n) if (i%4 == 4)]]

In [19]:
n = 5
[i for i in range(n) if (i%4 == 1)]

[1]

In [ ]:
def create_group(op_l, ien, op_r, start):
    block = [op_l, ien, op_r]

    ops = [id_mpo] * n

    pos = start
    while pos + 3 <= n:
        ops[pos:pos+3] = block
        pos += 3
        if pos + 1 <= n:
            ops[pos+1] = id_mpo
            pos += 1
    return ops

def create_A_group(n, site_i_half, site_ip2_half):
    id_mpo = Id.reshape((1,1,3,3))
    mpo_A = create_group(site_i_half, id_enlarged_mpo, site_ip2_half, 0)
    mpo_B = create_group(site_i_half, id_enlarged_mpo, site_ip2_half, 1)
    mpo_C = create_group(site_i_half, id_enlarged_mpo, site_ip2_half, 2)
    mpo_D = create_group(site_i_half, id_enlarged_mpo, site_ip2_half, 3)
    

In [44]:
from sympy import *

i, ien, Al, Ar = symbols("I I_e A_l A_r", commutative=False)

n = 11
start = 2
def shifted_blocks(n, start, Al, ien, Ar, i):
    block = [Al, ien, Ar]

    ops = [i] * n

    pos = start
    while pos + 3 <= n:
        ops[pos:pos+3] = block
        pos += 3
        if pos + 1 <= n:
            ops[pos+1] = i
            pos += 1
    return ops

display(shifted_blocks(n, start, Al, ien, Ar, i))

[I, I, A_l, I_e, A_r, I, A_l, I_e, A_r, I, I]

In [64]:
n = 9
mask_id_A = [(i%4 == 1) if (i+1)<n else False for i in range(n)]
mask_Cl = [(i%4 == 1) if (i+2)<n else False for i in range(n)]
mask_id_B = [(i%4 == 3) if (i+1)<n else False for i in range(n)]
mask_Dl = [(i%4 == 3) if (i+2)<n else False for i in range(n)]
print(mask_id_A)
print(mask_Cl)
print(mask_id_B)
print(mask_Dl)

[False, True, False, False, False, True, False, False, False]
[False, True, False, False, False, True, False, False, False]
[False, False, False, True, False, False, False, True, False]
[False, False, False, True, False, False, False, False, False]
